In [ ]:

import numpy as np
import time


# ==========================================
# A. PRESERVATION DU VOISINAGE
# ==========================================

def plus_proches_voisins(X_reference, X_queries, k=5):
    """
    Pour chaque requete, trouve ses k plus proches
    voisins dans X_reference.
    """

    # Distances euclidiennes
    distances = np.sqrt(
        np.sum(
            (X_queries[:, None, :] - X_reference[None, :, :])**2,
            axis=2
        )
    )

    voisins = np.argsort(distances, axis=1)[:, :k]

    return voisins


def recall_at_k(
    X_train_original,
    X_test_original,
    X_train_reduit,
    X_test_reduit,
    k=5
):
    """
    Compare les voisins dans l'espace original
    et dans l'espace reduit.
    """

    voisins_originaux = plus_proches_voisins(
        X_train_original, X_test_original, k
    )

    voisins_reduits = plus_proches_voisins(
        X_train_reduit, X_test_reduit, k
    )

    scores = []

    for i in range(len(X_test_original)):

        intersection = np.intersect1d(
            voisins_originaux[i],
            voisins_reduits[i]
        )

        score = len(intersection) / k
        scores.append(score)

    return np.mean(scores)


In [ ]:

# ==========================================
# B. DISTORSION DES DISTANCES
# ==========================================

def distorsion_distances(
    X_original,
    X_reduit,
    n_pairs=2000,
    seed=42
):
    """
    Mesure la distorsion relative des distances
    sur un echantillon de paires de points.
    """

    rng = np.random.default_rng(seed)

    n = len(X_original)

    # Generer des paires aleatoires
    i = rng.integers(0, n, size=n_pairs)
    j = rng.integers(0, n, size=n_pairs)

    # Eviter les paires identiques
    mask = i != j
    i = i[mask]
    j = j[mask]

    # Distances originales
    d_original = np.linalg.norm(
        X_original[i] - X_original[j],
        axis=1
    )

    # Distances apres reduction
    d_reduit = np.linalg.norm(
        X_reduit[i] - X_reduit[j],
        axis=1
    )

    # Distorsion relative
    distorsions = np.abs(
        d_reduit - d_original
    ) / d_original

    return {
        "distorsion_moyenne": np.mean(distorsions),
        "distorsion_mediane": np.median(distorsions),
        "distorsion_95": np.percentile(distorsions, 95)
    }


In [ ]:

# ==========================================
# C. RECONSTRUCTION
# ==========================================

def erreur_reconstruction(X_original, X_reconstruit):

    erreur = (
        np.linalg.norm(
            X_original - X_reconstruit,
            ord="fro"
        )**2
        /
        np.linalg.norm(X_original, ord="fro")**2
    )

    return erreur


def reconstruire_projection_gaussienne(Z, R, mean):
    """
    Reconstruction par pseudo-inverse.
    """

    R_pseudo_inverse = np.linalg.pinv(R)

    X_reconstruit = Z @ R_pseudo_inverse + mean

    return X_reconstruit


## Visualiser les images originales et reconstruites

In [ ]:
import matplotlib.pyplot as plt

# Reconstruction ACP
X_test_reconstructed = (
    Z_test_pca @ model_pca["components"].T
    + model_pca["mean"]
)

# Reconstruction projection aleatoire
X_test_random_reconstructed = reconstruire_projection_gaussienne(
    Z_test_random,
    model_random["R"],
    mean_train
)

fig, axes = plt.subplots(3, 5, figsize=(10, 6))

for i in range(5):

    axes[0, i].imshow(
        X_test[i].reshape(8, 8),
        cmap="gray"
    )
    axes[0, i].set_title("Originale")

    axes[1, i].imshow(
        X_test_reconstructed[i].reshape(8, 8),
        cmap="gray"
    )
    axes[1, i].set_title("ACP")

    axes[2, i].imshow(
        X_test_random_reconstructed[i].reshape(8, 8),
        cmap="gray"
    )
    axes[2, i].set_title("Projection aleatoire")

    for j in range(3):
        axes[j, i].axis("off")

plt.tight_layout()
plt.show()

## 6.2 Code de régression logistique

In [ ]:

# ==========================================
# D. REGRESSION LOGISTIQUE MULTINOMIALE
# ==========================================

def softmax(scores):

    scores = scores - np.max(
        scores, axis=1, keepdims=True
    )

    exp_scores = np.exp(scores)

    return exp_scores / np.sum(
        exp_scores, axis=1, keepdims=True
    )


def entrainer_logistic(
    X_train,
    y_train,
    alpha=0.001,
    learning_rate=0.1,
    epochs=1000
):
    """
    Regression logistique multinomiale
    avec regularisation L2.

    Standardisation apprise sur train uniquement.
    """

    n, d = X_train.shape

    classes = np.unique(y_train)
    nb_classes = len(classes)

    # Standardisation
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    std[std < 1e-12] = 1.0

    X = (X_train - mean) / std

    # Encodage des labels
    y_encoded = np.searchsorted(classes, y_train)

    Y = np.eye(nb_classes)[y_encoded]

    # Initialisation
    W = np.zeros((d, nb_classes))
    b = np.zeros(nb_classes)

    # Descente de gradient
    for epoch in range(epochs):

        scores = X @ W + b

        probabilities = softmax(scores)

        error = probabilities - Y

        grad_W = (X.T @ error) / n + alpha * W
        grad_b = np.mean(error, axis=0)

        W -= learning_rate * grad_W
        b -= learning_rate * grad_b

    return {
        "W": W,
        "b": b,
        "mean": mean,
        "std": std,
        "classes": classes
    }


def predire_logistic(X, model):

    X_scaled = (
        X - model["mean"]
    ) / model["std"]

    scores = X_scaled @ model["W"] + model["b"]

    probabilities = softmax(scores)

    indices = np.argmax(probabilities, axis=1)

    return model["classes"][indices]


## 6.3 Accuracy et Macro-F1


In [1]:

# ==========================================
# E. METRIQUES DE CLASSIFICATION
# ==========================================

def calculer_accuracy(y_true, y_pred):

    return np.mean(y_true == y_pred)


def calculer_macro_f1(y_true, y_pred):

    classes = np.unique(y_true)

    f1_scores = []

    for c in classes:

        TP = np.sum((y_true == c) & (y_pred == c))
        FP = np.sum((y_true != c) & (y_pred == c))
        FN = np.sum((y_true == c) & (y_pred != c))

        precision = TP / (TP + FP) if TP + FP > 0 else 0
        recall = TP / (TP + FN) if TP + FN > 0 else 0

        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = 0

        f1_scores.append(f1)

    return np.mean(f1_scores)


def evaluer_classification(
    X_train,
    y_train,
    X_test,
    y_test
):
    # Temps d'entrainement
    start = time.perf_counter()

    model = entrainer_logistic(
        X_train,
        y_train,
        alpha=0.001,
        learning_rate=0.1,
        epochs=1000
    )

    temps_entrainement = time.perf_counter() - start

    # Temps de prediction
    start = time.perf_counter()

    predictions = predire_logistic(X_test, model)

    temps_prediction = time.perf_counter() - start

    return {
        "accuracy": calculer_accuracy(y_test, predictions),
        "macro_f1": calculer_macro_f1(y_test, predictions),
        "temps_entrainement": temps_entrainement,
        "temps_prediction": temps_prediction
    }


## 7. C — Choisir la dimension q avec validation

In [ ]:

# ==========================================
# F. SELECTION DE q AVEC VALIDATION
# ==========================================

def selectionner_q(
    X_train,
    y_train,
    X_val,
    y_val,
    method,
    dimensions=[2, 5, 10, 20, 40],
    seed=42
):

    resultats = []

    mean_train = np.mean(X_train, axis=0)

    for q in dimensions:

        if method == "ACP":

            model = acp_directe_fit(X_train, q)

            Z_train = acp_directe_transform(
                X_train, model
            )

            Z_val = acp_directe_transform(
                X_val, model
            )

        elif method == "Random":

            model = projection_gaussienne_fit(
                X_train.shape[1],
                q,
                seed=seed
            )

            Z_train = (X_train - mean_train) @ model["R"]
            Z_val = (X_val - mean_train) @ model["R"]

        else:
            raise ValueError("Methode inconnue")

        # Evaluation sur validation
        model_logistic = entrainer_logistic(
            Z_train,
            y_train,
            alpha=0.001,
            learning_rate=0.1,
            epochs=1000
        )

        predictions = predire_logistic(
            Z_val,
            model_logistic
        )

        score = calculer_macro_f1(
            y_val,
            predictions
        )

        resultats.append({
            "q": q,
            "macro_f1_validation": score
        })

    meilleur = max(
        resultats,
        key=lambda r: r["macro_f1_validation"]
    )

    return meilleur, resultats


## 8. D — Comparer les trois représentations